# 02 · Databanken verkennen en aanvullen

Een weegtoestel en een voederstation registreren veel bezoeken. Door die gegevens in een databank te bewaren, kun je gericht records opzoeken, aantallen controleren en de gegevens later combineren. Voor je ermee rekent, moet je weten hoe de databank is opgebouwd en wat de velden betekenen.

Je opent in deze oefening je eigen databank op je laptop met **DB Browser for SQLite**. Je onderzoekt de tabellen, maakt een kleine nieuwe tabel en neemt vier rijen uit Excel over. Daarna upload je de bewerkte databank naar Codespaces en controleer je of je wijzigingen correct zijn aangekomen.

**Nodig:** je bestaande Codespace en eigen branch uit notebook 01, het opgeslagen bestand `gegevens/inlevering/01_werkplekcontrole.json`, je laptop en een programma waarmee je een `.xlsx`-bestand kunt lezen (Excel of LibreOffice Calc).

**Route:** rol ophalen → software installeren → databank downloaden en onderzoeken → oefentabel maken → vier rijen toevoegen → opslaan en uploaden → controleren en bewaren.

Voer de codecellen één voor één uit. Tussen de cellen voer je ook handelingen op je laptop uit. De voorbeelduitvoer hoort bij de rol gewichten; jouw eigen rol wordt automatisch uit notebook 01 opgehaald.

## 1. Je werkplek en rol ophalen

**Uitvoeren:** in Codespaces (Notebook 02)

Hervat je bestaande Codespace via [github.com/codespaces](https://github.com/codespaces) en open `notebooks/02_databanken_verkennen_en_aanvullen.ipynb`. Selecteer dezelfde Python-kernel als in notebook 01.

Je hebt je rol al opgeslagen. Door die keuze opnieuw in te lezen, gebruik je in beide notebooks dezelfde databank. De cel controleert ook of de opgeslagen instellingen bij je actieve branch horen.

In [1]:
from pathlib import Path
from contextlib import closing
import subprocess
import sys
import re
import json
import sqlite3
import pandas as pd

resultaat = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if resultaat.returncode:
    raise RuntimeError("Open dit notebook vanuit de repository in Codespaces.")
hoofdmap = Path(resultaat.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch):
    raise RuntimeError("Selecteer eerst je eigen student/voornaam_achternaam-branch uit notebook 01.")
instellingenpad = hoofdmap / "gegevens/inlevering/01_werkplekcontrole.json"
if not instellingenpad.is_file():
    raise FileNotFoundError("Voer eerst de controlebestandcel van notebook 01 uit en bewaar het resultaat.")
instellingen = json.loads(instellingenpad.read_text(encoding="utf-8"))
if instellingen.get("branch") != branch or instellingen.get("deelnemerscode") != branch.removeprefix("student/"):
    raise ValueError("De instellingen horen bij een andere branch. Voer notebook 01 op je eigen branch opnieuw uit.")
rol = instellingen.get("rol")
bestanden = {"gewichten": "asserva.db", "voederopname": "schauer.db"}
if rol not in bestanden or instellingen.get("bronbestand") != bestanden[rol]:
    raise ValueError("Controleer je rol en bronbestand in notebook 01.")
bestandsnaam = bestanden[rol]
origineel = hoofdmap / "gegevens/origineel" / bestandsnaam
invoermap = hoofdmap / "gegevens/werk/invoer"
invoermap.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(hoofdmap / "programmatuur"))
from databankcontrole import openen, inspecteer, controleer_aanvulling, bewaar_werkbestand
print("Branch:", branch)
print("Rol:", rol)
print("Jouw databank:", bestandsnaam)
print("Upload je bewerkte kopie straks naar: gegevens/werk/invoer/")

Branch: student/proefdocent
Rol: gewichten
Jouw databank: asserva.db
Upload je bewerkte kopie straks naar: gegevens/werk/invoer/


## 2. SQL, SQLite en DB Browser

**Uitvoeren:** op je laptop (browser en installatieprogramma)

Een **databank** bewaart gegevens volgens een afgesproken structuur. In een relationele databank staan ze in tabellen: een **record** is één rij, een **kolom** beschrijft een eigenschap van die rij. Een bezoek kan bijvoorbeeld een identificatie, datum en gewicht hebben.

**SQL** is de taal waarmee je gegevens opvraagt en tabellen aanpast. **SQLite** is de software die deze opdrachten uitvoert en de databank in één bestand bewaart. **DB Browser for SQLite** is het gratis grafische programma waarmee je dat bestand opent en bewerkt. Je installeert hier DB Browser; SQLite is daarin al aanwezig.

### Installeren

1. Open de [officiële downloadpagina](https://sqlitebrowser.org/dl/).
2. **Windows:** kies het installatieprogramma voor jouw Windows-versie, meestal de 64-bit installer. Open het gedownloade installatiebestand en volg de stappen. Start daarna **DB Browser for SQLite**, niet DB Browser for SQLCipher.
3. **macOS:** kies de macOS-download, open het `.dmg`-bestand en sleep DB Browser naar **Applications**. Open het daarna vanuit Applications.
4. Je ziet een venster met onder andere **Open Database**, **Database Structure**, **Browse Data** en **Execute SQL**.

De schermafbeeldingen in deze oefening zijn gemaakt met DB Browser for SQLite 3.13.1 op Windows, met Engelse menunamen. Op macOS hebben de overeenkomstige menu's en tabbladen dezelfde functie.

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;"><strong>Tip</strong><br>Heb je geen installatierechten op Windows, dan is op de officiële downloadpagina ook een ZIP-versie zonder installatie beschikbaar. Pak die volledig uit en open DB Browser for SQLite vanuit die map. Laat bij een blokkering door instellingsbeheer de toegestane installatie nakijken.</div>

## 3. Je databank en oefenbestand downloaden

**Uitvoeren:** in Codespaces (Explorer) en op je laptop (bestandenbeheer)

Je maakt een eigen werkversie, zodat het aangeleverde startbestand beschikbaar blijft als vergelijkingspunt. De bestanden `asserva.db` en `schauer.db` zijn direct te openen SQLite-databanken. Ze bevatten de eerste 28 kalenderdagen: **24 augustus tot en met 20 september 2023**.

| Jouw rol | Download dit bestand | Inhoud van `metingen` |
|---|---|---|
| gewichten | `gegevens/origineel/asserva.db` | 80.028 records met gewichtsgegevens |
| voederopname | `gegevens/origineel/schauer.db` | 146.649 records met voederopnamegegevens |

1. Maak op je laptop een map **`practicum_iot`** en daarin een map **`werk`**.
2. Open in Codespaces links **Explorer → gegevens → origineel**. Klik met de rechtermuisknop op jouw `.db`-bestand en kies **Download**.
3. Verplaats het gedownloade bestand naar `practicum_iot/werk/`. Behoud de naam `asserva.db` of `schauer.db`. Staat er `(1)` in de naam, controleer dan welke download je wilt gebruiken en geef die de juiste naam.
4. Download uit `gegevens/oefeningen/` ook **`oefenhokken.xlsx`** en **`oefenhokken.csv`** naar `practicum_iot/`.
5. Start DB Browser, kies **Open Database** en open de databank uit jouw lokale map `werk`.

Het oorspronkelijke bestand blijft in Codespaces onder `gegevens/origineel/` staan. De kopie op je laptop is degene die je nu bewerkt.

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;"><strong>Tip: een .db-bestand of een SQL-export</strong><br>Een <code>.db</code>-bestand bevat de databank zelf: kies <strong>Open Database</strong>. Een <code>.sql</code>-export bevat tekst met SQL-opdrachten en moet eerst via <strong>File → Import → Database from SQL file…</strong> worden ingelezen. Dat werkt alleen als de opdrachten geschikt zijn voor SQLite. In deze oefening ontvang je al een <code>.db</code>-bestand en gebruik je de importstap niet.</div>

## 4. De structuur en meetgegevens onderzoeken

**Uitvoeren:** op je laptop (DB Browser)

De structuur bepaalt wat iedere rij voorstelt. Een **primaire sleutel** identificeert een record uniek. In de tabel `metingen` is dat `ROW_ID`. Eén dier kan meerdere bezoeken hebben; daarom kan de dieridentificatie `EID` hier geen unieke sleutel voor een bezoek zijn.

1. Open **Database Structure** en klap **Tables** open.
2. Je ziet **`metingen`** en **`metadata`**. Klap `metingen` open om de kolommen te bekijken.
3. Zoek `ROW_ID`, `EID`, `DATE` en jouw meetveld: **`VBW`** voor gewichten of **`VFI`** voor voederopname.
4. Bekijk het gegevenstype. **TEXT** bewaart tekst, **REAL** een getal met mogelijke decimalen en **INTEGER** een geheel getal. Bij `ROW_ID` staan ook **PRIMARY KEY** en **NOT NULL**: de identificatie moet uniek en ingevuld zijn.
5. Ga naar **Browse Data**, selecteer `metingen` in de keuzelijst **Table** en bekijk enkele rijen. Schuif horizontaal om het meetveld en de kwaliteitsvelden te vinden.
6. Selecteer daarna `metadata`. Deze tabel beschrijft onder andere de rol, periode, eenheid en herkomst van de databank.

De oorspronkelijke kolomnamen zijn behouden, zodat je de koppeling met de aangeleverde bronbestanden kunt volgen. De bestandsnamen verwijzen naar de toestellen; dit is een onderwijsstructuur, geen officiële fabrikantendatabase.

![De twee tabellen van de startdatabank](../../afbeeldingen/schermafbeeldingen/02_databankstructuur.png)

*Figuur 1. Database Structure in DB Browser for SQLite 3.13.1 op Windows. Met het pijltje naast een tabel klap je de kolommen open.*

| Veld | Betekenis | Bewaarwijze / eenheid |
|---|---|---|
| `ROW_ID` | Identificatie van het bronrecord | TEXT; 27 of 28 cijfers |
| `EID` | Dieridentificatie | TEXT; hetzelfde dier kan meerdere bezoeken hebben |
| `PEN` | Hokcode in de bron | TEXT |
| `DATE` | Kalenderdatum waarop de bronselectie is gebaseerd | TEXT, `JJJJ-MM-DD` |
| `VBW` | Lichaamsgewicht bij een gewichtsrecord | REAL, kg |
| `VFI` | Voederopname per bezoek | REAL, g |
| `DATAORIGIN` | Herkomstcode, bijvoorbeeld `imputed` | TEXT; broncode behouden |
| `FILTER`, `ERROR`, `FLAG_…` | Kwaliteitsinformatie uit de verwerking | TEXT; broncodes behouden |

De volledige veldtoelichting staat in [het gegevenswoordenboek](../../gegevens/gegevenswoordenboek.md).

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;"><strong>Uitleg</strong><br>Een recordnummer is een herkenningscode, geen hoeveelheid. Je hoeft er niet mee te rekenen. Met TEXT blijven alle 27 of 28 cijfers behouden. Excel kan lange codes afronden wanneer het ze als getal behandelt. Een afgekorte weergave door een smalle kolom is iets anders dan werkelijk verloren cijfers.</div>

**Vraag 1.** Welke twee tabellen bevat jouw startbestand? Hoeveel kolommen heeft `metingen`, wat is de primaire sleutel en waarom is `EID` niet geschikt als primaire sleutel?

**Jouw antwoord (Vraag 1):**

*Voeg hier je antwoord in. Tip: noteer jouw rol en zoek de kolommen in Database Structure.*

**Antwoord 1**

Beide bestanden bevatten `metingen` en `metadata`. Bij gewichten heeft `metingen` 22 kolommen, bij voederopname 24. `ROW_ID` is de primaire sleutel. `EID` komt terug bij verschillende bezoeken van hetzelfde dier en is daarom niet uniek per rij.

## 5. Met SQL gericht gegevens opvragen

**Uitvoeren:** op je laptop (DB Browser → Execute SQL)

Met een **query** vraag je precies de gegevens op die je nodig hebt. Je kunt daarmee ook controleren of je het volledige bestand hebt geopend. `SELECT` bepaalt wat je wilt zien, `FROM` uit welke tabel en `WHERE` welke rijen je selecteert. `COUNT(*)` telt rijen. `MIN` en `MAX` zoeken de kleinste en grootste waarde.

1. Open het tabblad **Execute SQL**.
2. Kopieer onderstaande opdracht naar het grote tekstvak.
3. Klik op **Execute SQL** (het afspeelknopje, of **F5**). Selecteer geen losse regel: zonder tekstselectie voert DB Browser de volledige opdracht uit.
4. Lees de resultaatrij onder het tekstvak. De query verandert de databank niet.

```sql
SELECT COUNT(*) AS aantal_records,
       MIN(DATE) AS eerste_datum,
       MAX(DATE) AS laatste_datum
FROM metingen;
```

![Query met aantal en datumgrenzen](../../afbeeldingen/schermafbeeldingen/02_query.png)

*Figuur 2. Het SQL-tekstvak, de uitvoerknop en het queryresultaat voor de gewichtsdatabank, in DB Browser 3.13.1 op Windows.*

Je verwacht 80.028 of 146.649 records, afhankelijk van je rol, met als uiterste datums `2023-08-24` en `2023-09-20`. Vervang de tekst daarna door deze tweede query:

```sql
SELECT ROW_ID, EID, DATE, FILTER
FROM metingen
ORDER BY DATE, ROW_ID
LIMIT 5;
```

`ORDER BY` sorteert het resultaat; `LIMIT 5` toont alleen de eerste vijf rijen van dat resultaat. De overige records blijven in de databank staan.

**Vraag 2.** Welke aantallen en datums geeft jouw eerste query? Betekent het resultaat van de tweede query dat de databank nog maar vijf meetrecords bevat?

**Jouw antwoord (Vraag 2):**

*Voeg hier je antwoord in. Tip: vermeld concrete uitkomsten en leg de functie van LIMIT uit.*

**Antwoord 2**

Gewichten: 80.028 records; voederopname: 146.649 records. Beide lopen van 24 augustus tot en met 20 september 2023. `LIMIT 5` begrenst alleen het getoonde queryresultaat. De databank blijft volledig.

**Vraag 3.** Welke eenheid hoort bij jouw meetveld? Wat betekent het wanneer een gewicht als `imputed` is gemarkeerd, en waarom bewaar je zo’n markering?

**Jouw antwoord (Vraag 3):**

*Voeg hier je antwoord in. Tip: gebruik metadata en het gegevenswoordenboek; onderscheid een gemeten waarde van een berekende aanvulling.*

**Antwoord 3**

`VBW` staat in kg en `VFI` in gram per bezoek. `imputed` duidt op een ingevuld of geschat gewicht uit de bronverwerking. Door die herkomst te bewaren, kun je bij een latere analyse onderscheid maken tussen zulke waarden en rechtstreekse metingen. De markering rechtvaardigt op zichzelf geen automatische verwijdering.

## 6. Een eigen oefentabel maken

**Uitvoeren:** op je laptop (DB Browser → Database Structure)

Meetgegevens krijgen vaak context, bijvoorbeeld een hokcode en een beschrijving. Je oefent dat principe met **`oefen_hokken`**: een kleine tabel met fictieve afdelingsnamen en een controlevolgorde. De records zijn uitsluitend oefenmateriaal en beschrijven niet de werkelijke proefinrichting.

1. Kies **Database Structure → Create Table**.
2. Vul bij **Table** de naam **`oefen_hokken`** in.
3. Klik op **Add** om een veld toe te voegen. Maak onderstaande vier velden, in deze volgorde. Dubbelklik op de naam of het type om het te wijzigen.
4. Vink bij `hok` zowel **PK** (Primary Key) als **NN** (Not Null) aan. Laat **AI** (Auto Increment) uit.
5. Controleer de veldnamen en types en klik op **OK**.

| Name | Type | NN | PK |
|---|---|---|---|
| `hok` | TEXT | aangevinkt | aangevinkt |
| `afdeling` | TEXT | uit | uit |
| `controlevolgorde` | INTEGER | uit | uit |
| `notitie` | TEXT | uit | uit |

DB Browser vertaalt deze keuzes naar een SQL-opdracht. Onderaan het venster kun je die opdracht bekijken. Zo zie je de relatie tussen de grafische bediening en SQL.

![De vier kolommen en sleutelinstellingen](../../afbeeldingen/schermafbeeldingen/02_tabeldefinitie.png)

*Figuur 3. De ingevulde definitie van oefen_hokken in DB Browser 3.13.1 op Windows. Bij hok zijn NN en PK aangevinkt; de andere velden hebben de hierboven opgegeven types.*

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;"><strong>Tip: verderwerken</strong><br>Bestaat <code>oefen_hokken</code> al omdat je de oefening hervat? Controleer dan de kolommen met <strong>Modify Table</strong> en werk verder met die tabel. Maak geen tweede tabel met bijna dezelfde naam.</div>

## 7. Vier Excel-rijen handmatig toevoegen

**Uitvoeren:** op je laptop (Excel of Calc en DB Browser → Browse Data)

Door enkele rijen zelf in te voeren, zie je hoe een werkbladrij een databankrecord wordt. Het kleine oefenbestand maakt het mogelijk om de inhoud volledig na te kijken.

1. Open **`oefenhokken.xlsx`**. Op het blad **Oefenhokken** staan vier gegevensrijen onder de kolomkoppen.
2. Ga in DB Browser naar **Browse Data** en selecteer `oefen_hokken` bij **Table**.
3. Klik op **New Record** (of **Insert Record**, afhankelijk van de versie).
4. Dubbelklik op een cel. In deze interface bewerk je de waarde rechts onder **Edit Database Cell**. Typ de waarde en klik daar op **Apply**; controleer daarna of ze in de tabel verschijnt. Herhaal dit voor de vier waarden van de eerste Excel-rij. Gebruik bij de volgorde het getal `1`, zonder aanhalingstekens. Controleer ook de hokcode, zelfs als DB Browser die alvast invult.
5. Voeg op dezelfde manier de andere drie records toe. Neem alleen de cellen **A2:D5** over; de toelichting onderaan het werkblad is geen record.
6. Vergelijk alle vier rijen met Excel. Staat de tabel bij hervatten al gevuld, verbeter dan alleen afwijkingen en voeg de rijen niet opnieuw toe.

Je gebruikt `hok` als unieke code. Wanneer je dezelfde hokcode nogmaals invoert, kan de primaire sleutel die dubbele rij tegenhouden. Een database kan hierdoor fouten opvangen die in een gewoon werkblad gemakkelijk onopgemerkt blijven.

### De inhoud controleren

**Uitvoeren:** op je laptop (DB Browser → Execute SQL)

Vervang de vorige query door:

```sql
SELECT hok, afdeling, controlevolgorde, notitie
FROM oefen_hokken
ORDER BY controlevolgorde;
```

Voer de query uit. Je ziet vier rijen in dezelfde volgorde als in Excel. Controleer de spelling en de volgordegetallen.

**Zelf proberen:** pas de query aan zodat alleen **Oefenafdeling B** verschijnt. Zet de selectie met `WHERE` tussen `FROM` en `ORDER BY`. De onderstaande codecel is alleen een plaats om jouw SQL-tekst te bewaren; de query zelf voer je in DB Browser uit.

In [2]:
# Bewaar hier jouw SQL-opdracht als tekst tussen de drie aanhalingstekens.
# Begin met SELECT ... FROM oefen_hokken.
# Voeg WHERE afdeling = 'Oefenafdeling B' toe vóór ORDER BY.
mijn_query = """
"""

In [3]:
mijn_query = """
SELECT hok, afdeling, controlevolgorde, notitie
FROM oefen_hokken
WHERE afdeling = 'Oefenafdeling B'
ORDER BY controlevolgorde;
"""
print(mijn_query)


SELECT hok, afdeling, controlevolgorde, notitie
FROM oefen_hokken
WHERE afdeling = 'Oefenafdeling B'
ORDER BY controlevolgorde;



**Vraag 4.** Welke hokcodes verschijnen bij je filter op Oefenafdeling B? Zijn de andere twee rijen verwijderd?

**Jouw antwoord (Vraag 4):**

*Voeg hier je antwoord in. Tip: noem de codes en maak onderscheid tussen selecteren en verwijderen.*

**Antwoord 4**

De hokcodes `3` en `4` verschijnen. De rijen met codes `1` en `2` blijven opgeslagen. `WHERE` selecteert welke rijen in dit queryresultaat verschijnen; de SELECT-opdracht verwijdert geen records.

## 8. Het alternatief voor veel rijen: CSV-import

**Uitvoeren:** op je laptop (DB Browser)

Bij vier rijen kun je handmatig controleren wat je invoert. Bij duizenden rijen is een import geschikter. **CSV** bewaart een tabel als tekst, met een scheidingsteken tussen de velden. DB Browser leest CSV rechtstreeks; een Excel-werkboek moet je daarvoor eerst naar CSV exporteren.

Je vergelijkt beide methoden met dezelfde vier oefenrijen. Importeer ze in een **aparte vergelijkingstabel**, zodat je de handmatige invoer behoudt.

1. Kies **File → Import → Table from CSV file…** en open het meegeleverde `oefenhokken.csv`.
2. Geef de doeltabel de naam **`oefen_hokken_csv`**. Gebruik niet de naam `oefen_hokken`.
3. Vink **Column names in first line** aan, kies een **komma** als scheidingsteken en **UTF-8** als tekstcodering.
4. Controleer het voorbeeld: vier kolommen, met de namen `hok`, `afdeling`, `controlevolgorde` en `notitie` bovenaan. Bevestig de import.
5. Bekijk `oefen_hokken_csv` via **Browse Data** en vergelijk de vier rijen met je handmatige tabel.

Bij hervatten: bestaat `oefen_hokken_csv` al, bekijk die tabel dan opnieuw in plaats van nogmaals te importeren. De automatische import kan andere gegevenstypen kiezen en maakt niet vanzelf dezelfde primaire sleutel. Bekijk dat verschil in **Database Structure**.

![CSV-instellingen met kolomkoppen en vier rijen](../../afbeeldingen/schermafbeeldingen/02_csv_import.png)

*Figuur 4. CSV-import in DB Browser 3.13.1 op Windows. De doeltabel heeft een eigen naam, de eerste regel bevat kolomkoppen en het voorbeeld toont vier records.*

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;"><strong>Tip: zelf exporteren vanuit Excel</strong><br>Je kunt de vier gegevensrijen en hun koppen ook naar een apart werkblad kopiëren en dat als CSV UTF-8 opslaan. Kies in DB Browser het scheidingsteken dat je in het voorbeeld ziet: Nederlandstalige Excel-instellingen gebruiken soms een puntkomma. Een CSV bevat één tabel en bewaart geen werkbladopmaak.</div>

## 9. Opslaan, sluiten en uploaden

**Uitvoeren:** op je laptop (DB Browser), daarna in Codespaces (Explorer)

DB Browser houdt bewerkingen bij voordat je ze definitief naar het databasebestand schrijft. Door eerst op te slaan en de database te sluiten, upload je een consistente kopie met alle wijzigingen.

1. Klik op **Write Changes** bovenaan DB Browser. **Save Project** is iets anders: dat bewaart venster- en projectinstellingen, niet de vervanging voor Write Changes.
2. Kies **Close Database**.
3. Open dezelfde lokale databank nogmaals met **Open Database**. Controleer via Browse Data dat de vier rijen in `oefen_hokken` er nog staan. Sluit de databank daarna opnieuw.
4. Ga terug naar Codespaces. Open in Explorer **gegevens → werk → invoer**.
5. Klik met de rechtermuisknop op **invoer**, kies **Upload…** en selecteer de gesloten databank uit `practicum_iot/werk/` op je laptop. Je kunt het bestand ook naar die map slepen.
6. Controleer dat de bestandsnaam exact overeenkomt met de naam die stap 1 van dit notebook afdrukte.

De map `invoer` is een ontvangstmap. In de volgende stap vergelijken we de upload met het origineel voordat die jouw werkdatabase wordt. Upload dus naar **`gegevens/werk/invoer/`**, niet naar `gegevens/origineel/`.

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;"><strong>Controle vóór je verdergaat</strong><br>De lokale database is opgeslagen en gesloten. In Codespaces staat jouw bewerkte <code>asserva.db</code> of <code>schauer.db</code> onder <code>gegevens/werk/invoer/</code>. Voer pas daarna de volgende cel uit.</div>

## 10. De overgebrachte databank controleren

**Uitvoeren:** in Codespaces (Notebook 02)

Een juiste bestandsnaam bewijst nog niet dat je de juiste versie hebt overgebracht. De aangeleverde controlecode leest de database, vergelijkt alle oorspronkelijke meetrecords en metadata, en controleert de vier nieuwe oefenrijen. Ze opent de upload alleen om te lezen; een ontbrekend bestand wordt niet stilzwijgend als een lege database aangemaakt.

In [4]:
kandidaat = invoermap / bestandsnaam
controle = controleer_aanvulling(origineel, kandidaat, rol)
print("De meetgegevens en metadata zijn ongewijzigd.")
print("Meetrecords:", controle["aantal_metingen"])
print("Periode:", controle["eerste_datum"], "tot en met", controle["laatste_datum"])
print("Oefenrecords:", controle["aantal_oefenrijen"])
with closing(openen(kandidaat)) as db:
    display(pd.read_sql_query("SELECT * FROM oefen_hokken ORDER BY controlevolgorde", db))

De meetgegevens en metadata zijn ongewijzigd.
Meetrecords: 80028
Periode: 2023-08-24 tot en met 2023-09-20
Oefenrecords: 4


,hok,afdeling,controlevolgorde,notitie
0,1,Oefenafdeling A,1,Start aan de deur
1,2,Oefenafdeling A,2,Controleer het label
2,3,Oefenafdeling B,3,Noteer de observatie
3,4,Oefenafdeling B,4,Rond de controle af


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;"><strong>Interpretatie</strong><br>Je ziet vier oefenrecords en het oorspronkelijke aantal meetrecords voor jouw rol. Je hebt de databank uitgebreid met contextgegevens; de meettabel is intact gebleven. Verschijnt een foutmelding, herstel dan de aangegeven stap en upload de opnieuw opgeslagen kopie.</div>

### De gecontroleerde kopie als werkdatabase bewaren

Voor de volgende oefeningen krijgt de gecontroleerde database een vaste plaats: **`gegevens/werk/asserva.db`** of **`gegevens/werk/schauer.db`**. De cel gebruikt daarvoor de back-upfunctie van SQLite.

Als daar al een identieke werkdatabase staat, blijft die behouden. Staat er een andere versie, dan stopt de cel: je bekijkt eerst welke versie de nieuwste is. Zo kan een oude laptopkopie geen nieuwere werkdatabase vervangen.

In [5]:
werkdatabase = hoofdmap / "gegevens/werk" / bestandsnaam
verslag = bewaar_werkbestand(origineel, kandidaat, werkdatabase, rol)
print(verslag["overname"])
print("Werkdatabase:", werkdatabase.relative_to(hoofdmap))
with closing(openen(werkdatabase)) as db:
    display(pd.read_sql_query("PRAGMA table_info(oefen_hokken)", db))

gecontroleerde werkdatabase aangemaakt
Werkdatabase: gegevens/werk/asserva.db


,cid,name,type,notnull,dflt_value,pk
0,0,hok,TEXT,1,None,1
1,1,afdeling,TEXT,0,None,0
2,2,controlevolgorde,INTEGER,0,None,0
3,3,notitie,TEXT,0,None,0


**Vraag 5.** Waarom upload je eerst naar `gegevens/werk/invoer/` en controleer je daarna de inhoud? Waarom blijft je werkdatabase buiten Git?

**Jouw antwoord (Vraag 5):**

*Voeg hier je antwoord in. Tip: verbind dit met het verschil tussen je laptopkopie, werkdatabase en inleverbestanden.*

**Antwoord 5**

De ontvangstmap laat toe om de upload te controleren voordat die de vaste werkdatabase wordt. De vergelijking controleert ook de inhoud, niet alleen de naam. Werkdatabases veranderen tijdens de oefeningen en worden daarom buiten routinematige Git-commits gehouden. Het notebook en een klein controleverslag gaan wel naar GitHub. Voor behoud bij het verwijderen van een Codespace download je de werkdatabase apart.

## 11. Je resultaat bewaren en afsluiten

**Uitvoeren:** in Codespaces (Notebook 02 en Source Control)

Je bewaart een klein controleverslag met je rol, aantallen en bestandslocatie. Zo kun je later aantonen welke stap gelukt is, zonder de hele werkdatabase telkens opnieuw naar GitHub te sturen.

In [6]:
inlevering = hoofdmap / "gegevens/inlevering/02_databankcontrole.json"
inlevering.parent.mkdir(parents=True, exist_ok=True)
resultaat = {
    "deelnemerscode": instellingen["deelnemerscode"],
    "branch": branch,
    "rol": rol,
    "werkdatabase": werkdatabase.relative_to(hoofdmap).as_posix(),
    "aantal_metingen": verslag["aantal_metingen"],
    "aantal_oefenrijen": verslag["aantal_oefenrijen"],
    "basis_vanaf": verslag["eerste_datum"],
    "basis_tot_en_met": verslag["laatste_datum"],
    "metingen_sha256": verslag["metingen_sha256"],
}
inlevering.write_text(json.dumps(resultaat, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
assert json.loads(inlevering.read_text(encoding="utf-8")) == resultaat
print("Bewaard:", inlevering.relative_to(hoofdmap))
print("Je kunt nu opslaan, committen en pushen.")

Bewaard: gegevens/inlevering/02_databankcontrole.json
Je kunt nu opslaan, committen en pushen.


<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;"><strong>Stop je Codespace na het practicum</strong><br>Je beschikbare gratis Codespaces-uren zijn beperkt. Bewaar je werk met de stappen hieronder en kies daarna <strong>Stop Current Codespace</strong>, of stop de werkplek via <a href="https://github.com/codespaces">github.com/codespaces</a>. Alleen de browsertab sluiten is niet hetzelfde als stoppen.</div>

1. Vul je antwoorden in en kies **File → Save All**.
2. Open **Source Control**. Selecteer dit notebook en `gegevens/inlevering/02_databankcontrole.json` met het **+**-teken naast elk bestand.
3. De werkdatabase onder `gegevens/werk/` hoort niet in de commitlijst te staan. De oorspronkelijke databank onder `gegevens/origineel/` hoort niet als gewijzigd te verschijnen.
4. Commit met bijvoorbeeld **Databank onderzocht en oefentabel toegevoegd** en kies **Push**.
5. Controleer op GitHub op je eigen branch of het notebook en het controleverslag zichtbaar zijn.
6. Download de gesloten werkdatabase uit `gegevens/werk/` naar een aparte map `practicum_iot/bewaard_na_02/` op je laptop. Zo heb je naast de Codespace ook een bewaarde kopie. Houd de mapnaam bij de kopie zodat je de versie herkent.
7. Stop je Codespace. Hervat bij de volgende sessie dezelfde werkplek.

### Eindcontrole

- [ ] Ik kan SQL, SQLite en DB Browser van elkaar onderscheiden.
- [ ] Ik heb tabellen, kolommen, primaire sleutel, eenheid en aantallen opgezocht.
- [ ] Mijn oefentabel bevat de vier juiste Excel-rijen.
- [ ] Ik kan met een WHERE-voorwaarde twee oefenrijen selecteren.
- [ ] Ik heb de CSV-import vergeleken met handmatige invoer.
- [ ] De controle in Codespaces slaagt en mijn vaste werkdatabase is bewaard.
- [ ] Mijn notebook en controleverslag staan op mijn eigen GitHub-branch.
- [ ] Mijn Codespace is gestopt.

## Hulp bij veelvoorkomende problemen

| Probleem | Wat controleer je? |
|---|---|
| Het notebook of de nieuwe bestanden ontbreken | Je branch heeft de nieuwe lesbestanden nog niet. Bewaar eerst je werk; laat de nieuwe bestanden aan je branch toevoegen. Maak niet zomaar een nieuwe lege werkplek. |
| Er ontbreekt een rolbestand | Doorloop de controlebestandcel van notebook 01 op je eigen branch. |
| DB Browser vraagt een wachtwoord | Open de gewone DB Browser for SQLite en het aangeleverde `.db`-bestand. De lesdatabase is niet versleuteld. |
| `no such table` bij een query | Controleer de geopende databank en de naam in de Table-keuzelijst. |
| De upload wordt niet gevonden | Controleer `gegevens/werk/invoer/`, de bestandsnaam en eventuele `(1)` in de downloadnaam. |
| De vier rijen ontbreken na upload | Open de laptopkopie, kies Write Changes, sluit de database en upload opnieuw. |
| Een primaire-sleutelfout bij invoer | De hokcode bestaat al. Verbeter de bestaande rij in plaats van een dubbele toe te voegen. |
| De controle meldt gewijzigde meetgegevens | Vergelijk met je originele download. Maak zo nodig een nieuwe lokale werkkopie en voeg alleen de oefentabel toe. Bewaar de afwijkende kopie apart als je die wilt onderzoeken. |
| De werkdatabase bestaat al en verschilt | Bewaar de bestaande werkdatabase apart en bepaal welke versie je nodig hebt. Verwijder geen bestand om de melding zomaar te laten verdwijnen. |
| Je werkt in een nieuwe Codespace | Je gepushte antwoorden zijn beschikbaar; upload je apart bewaarde database naar de ontvangstmap en voer de controles opnieuw uit. |

### Technische bronnen

- [DB Browser for SQLite: downloads](https://sqlitebrowser.org/dl/)
- [DB Browser: gebruikershandleiding](https://github.com/sqlitebrowser/sqlitebrowser/wiki)
- [SQLite: gegevenstypen](https://www.sqlite.org/datatype3.html)
- [SQLite: tabellen en beperkingen](https://www.sqlite.org/lang_createtable.html)
- [Python: SQLite-back-upfunctie](https://docs.python.org/3/library/sqlite3.html#sqlite3.Connection.backup)

De schermafbeeldingen zijn eigen opnames van de gebruikte Windows-interface. Ze zijn bijgesneden om alleen de relevante bediening te tonen.